In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("../dataset/cleaned_retail.csv")

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [ ]:
df['TotalPrice'] = (
    df['Quantity'] *
    df['UnitPrice']
)

In [ ]:
snapshot_date = df['InvoiceDate'].max()

snapshot_date

Timestamp('2011-12-09 12:50:00')

In [ ]:
rfm = df.groupby('CustomerID').agg({

    'InvoiceDate': lambda x:
        (snapshot_date - x.max()).days,

    'InvoiceNo': 'nunique',

    'TotalPrice': 'sum'

})

In [ ]:
rfm.columns = [
    'Recency',
    'Frequency',
    'Monetary'
]

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,325,1,77183.60
12347.0,1,7,4310.00
12348.0,74,4,1797.24
12349.0,18,1,1757.55
12350.0,309,1,334.40


In [ ]:
rfm.shape

(4338, 3)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

In [ ]:
rfm_scaled

array([[ 2.33457414e+00, -4.25096503e-01,  8.36301037e+00],
       [-9.05340320e-01,  3.54416797e-01,  2.51698905e-01],
       [-1.75359593e-01, -3.53398530e-02, -2.79878269e-02],
       ...,
       [-8.45341904e-01, -2.95177619e-01, -2.08214334e-01],
       [-8.85340848e-01,  1.52368675e+00, -3.51515172e-04],
       [-4.95351144e-01, -1.65258736e-01, -2.35311112e-02]],
      shape=(4338, 3))

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

rfm['Cluster'] = kmeans.fit_predict(
    rfm_scaled
)

In [ ]:
rfm.head()

,Recency,Frequency,Monetary,Cluster
CustomerID,,,,
12346.0,325,1,77183.60,1
12347.0,1,7,4310.00,0
12348.0,74,4,1797.24,0
12349.0,18,1,1757.55,0
12350.0,309,1,334.40,1


In [ ]:
rfm['Cluster'].value_counts()

Cluster
0    3231
1    1093
2      14
Name: count, dtype: int64

In [ ]:
rfm.groupby('Cluster').mean()

,Recency,Frequency,Monetary
Cluster,,,
0,39.983906,4.852677,2005.830431
1,245.023788,1.582800,629.357439
2,6.142857,80.214286,122748.792143


In [ ]:
cluster_map = {
    0: "Regular",
    1: "Low Value",
    2: "Premium"
}

rfm['CustomerSegment'] = rfm['Cluster'].map(
    cluster_map
)

In [ ]:
rfm.head()

,Recency,Frequency,Monetary,Cluster,CustomerSegment
CustomerID,,,,,
12346.0,325,1,77183.60,1,Low Value
12347.0,1,7,4310.00,0,Regular
12348.0,74,4,1797.24,0,Regular
12349.0,18,1,1757.55,0,Regular
12350.0,309,1,334.40,1,Low Value


In [ ]:
rfm['CustomerSegment'].value_counts()

CustomerSegment
Regular      3231
Low Value    1093
Premium        14
Name: count, dtype: int64

In [ ]:
rfm.to_csv(
    "../dataset/customer_segments.csv"
)

In [ ]:
rfm.to_csv(
    "../dataset/customer_segments.csv"
)

print("Customer Segments Saved ✅")

Customer Segments Saved ✅
